
# GP Geo CV Tuning

Nested geographic tuning workflow. The expensive cells are restartable: each `(param_id, fold_idx)` fit writes one CSV under `../eval_results/`, and reruns skip completed files.


In [ ]:

import json
import os
import subprocess
import sys
import time
from pathlib import Path

import jax.numpy as jnp
import jax.random as jr
import numpy as np
import pandas as pd

CWD = Path.cwd()
if CWD.name == "gridsearch":
    NOTEBOOK_DIR = CWD
elif (CWD / "gridsearch").exists() and CWD.name == "code":
    NOTEBOOK_DIR = CWD / "gridsearch"
elif (CWD / "code" / "gridsearch").exists():
    NOTEBOOK_DIR = CWD / "code" / "gridsearch"
else:
    NOTEBOOK_DIR = CWD
CODE_DIR = NOTEBOOK_DIR.parent

sys.path.append(str(CODE_DIR))

from gp import GaussianProcess
from gp_kernels import make_indoor_outdoor_mean, make_wifi_kernel
from eval_workflow import (
    completed_gp_results,
    load_fold_arrays,
    read_result_row,
    result_path,
    summarize_cv_results,
    write_result_row,
)


In [ ]:

SPLIT_ROOT = CODE_DIR / "eval_splits" / "geo_outer0"
GRID_PATH = CODE_DIR / "gridsearch" / "gp_geo_stage1_grid.csv"
STAGE1_RESULT_DIR = CODE_DIR / "eval_results" / "gp_stage1"
STAGE2_RESULT_DIR = CODE_DIR / "eval_results" / "gp_stage2"
FINAL_RESULT_PATH = CODE_DIR / "eval_results" / "gp_final_holdout.csv"

STAGE1_FOLDS = 3
STAGE2_FOLDS = 5
STAGE2_TOP_N = 25

N_CHAINS = 2
N_SAMPLES = 111
BURNIN = 10
THIN = 10
CALIBRATION_ITERS = 30
JITTER_FACTOR = 10
PREDICT_METHOD = "sequential"
KEY_SEED = 305


RUN_FITS_IN_SUBPROCESS = True
# Sequential chunking: one subprocess runs this many fits, exits to release GPU memory,
# then the notebook launches the next subprocess. This is not parallel.
FITS_PER_PROCESS = 20
GP_FIT_RUNNER = CODE_DIR / "run_gp_cv_fit.py"
PYTHON_EXECUTABLE = sys.executable


In [ ]:

grid = pd.read_csv(GRID_PATH)
grid.shape, grid.head()


In [ ]:

def format_seconds(seconds):
    seconds = int(round(float(seconds)))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    if hours:
        return f"{hours}h {minutes:02d}m {seconds:02d}s"
    if minutes:
        return f"{minutes}m {seconds:02d}s"
    return f"{seconds}s"


def print_loop_progress(stage_label, completed_count, total_count, loop_start, session_completed):
    elapsed = time.time() - loop_start
    avg = elapsed / max(session_completed, 1)
    remaining = max(total_count - completed_count, 0)
    eta = avg * remaining if session_completed else float("nan")
    eta_text = format_seconds(eta) if session_completed else "unknown"
    print(
        f"{stage_label} progress: {completed_count}/{total_count} complete; "
        f"session elapsed {format_seconds(elapsed)}; "
        f"avg/new fit {format_seconds(avg)}; ETA {eta_text}",
        flush=True,
    )


def count_completed_fits(param_grid, n_folds, result_dir):
    count = 0
    for _, param_row in param_grid.iterrows():
        for fold_idx in range(n_folds):
            out_path = result_path(result_dir, param_row["param_id"], fold_idx)
            count += read_result_row(out_path) is not None
    return int(count)


def pending_jobs(param_grid, n_folds, result_dir):
    jobs = []
    for _, param_row in param_grid.iterrows():
        param_id = int(param_row["param_id"])
        for fold_idx in range(n_folds):
            if read_result_row(result_path(result_dir, param_id, fold_idx)) is None:
                jobs.append({"param_id": param_id, "fold_idx": int(fold_idx)})
    return jobs


def job_chunks(jobs, chunk_size):
    for start in range(0, len(jobs), int(chunk_size)):
        yield jobs[start:start + int(chunk_size)]


def run_fit_chunk(stage_label, chunk, split_dir, result_dir):
    # subprocess.run is blocking, so chunks run sequentially: at most one GPU process.
    cmd = [
        PYTHON_EXECUTABLE,
        str(GP_FIT_RUNNER),
        "--grid-path", str(GRID_PATH),
        "--split-dir", str(split_dir),
        "--result-dir", str(result_dir),
        "--jobs-json", json.dumps(chunk),
        "--chains", str(N_CHAINS),
        "--samples", str(N_SAMPLES),
        "--burnin", str(BURNIN),
        "--thin", str(THIN),
        "--calibration-iters", str(CALIBRATION_ITERS),
        "--jitter-factor", str(JITTER_FACTOR),
        "--predict-method", PREDICT_METHOD,
        "--key-seed", str(KEY_SEED),
    ]
    env = os.environ.copy()
    env.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
    env.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")
    print(
        f"{stage_label}: launching one subprocess for {len(chunk)} fit(s): "
        f"param {chunk[0]['param_id']} fold {chunk[0]['fold_idx']} -> "
        f"param {chunk[-1]['param_id']} fold {chunk[-1]['fold_idx']}",
        flush=True,
    )
    subprocess.run(cmd, check=True, env=env)


def run_stage_loop(stage_label, param_grid, n_folds, split_dir, result_dir):
    total = len(param_grid) * n_folds
    completed_count = count_completed_fits(param_grid, n_folds, result_dir)
    jobs = pending_jobs(param_grid, n_folds, result_dir)
    loop_start = time.time()
    session_completed = 0
    print(f"Completed {completed_count} / {total} {stage_label.lower()} fits")

    if RUN_FITS_IN_SUBPROCESS:
        for chunk in job_chunks(jobs, FITS_PER_PROCESS):
            run_fit_chunk(stage_label, chunk, split_dir, result_dir)
            session_completed += len(chunk)
            completed_count += len(chunk)
            print_loop_progress(stage_label, completed_count, total, loop_start, session_completed)
        return

    for job in jobs:
        param_row = param_grid.loc[param_grid["param_id"] == job["param_id"]].iloc[0]
        print(f"{stage_label} param {job['param_id']} fold {job['fold_idx']}", flush=True)
        row = fit_evaluate_gp(param_row, split_dir, job["fold_idx"], result_dir)
        session_completed += 1
        completed_count += 1
        print(
            f"{stage_label} fit timing: total={format_seconds(row['total_elapsed'])}, "
            f"gibbs={format_seconds(row['fit_elapsed'])}, "
            f"predict={format_seconds(row['predict_elapsed'])}, mse={row['mse']:.4f}",
            flush=True,
        )
        print_loop_progress(stage_label, completed_count, total, loop_start, session_completed)


def fit_evaluate_gp(param_row, split_dir, fold_idx, result_dir):
    param_id = int(param_row["param_id"])
    out_path = result_path(result_dir, param_id, fold_idx)
    completed_row = read_result_row(out_path)
    if completed_row is not None:
        return completed_row

    total_start = time.time()
    arrays = load_fold_arrays(split_dir, fold_idx)
    X_train = jnp.asarray(arrays["X_train"])
    y_train = jnp.asarray(arrays["y_train"])
    obs_count_train = jnp.asarray(arrays["obs_count_train"])
    obs_sse_train = jnp.asarray(arrays["obs_sse_train"])
    X_test = jnp.asarray(arrays["X_test"])
    y_test = jnp.asarray(arrays["y_test"])

    m = make_indoor_outdoor_mean(X_train, y_train)
    K = make_wifi_kernel(
        ap_form=param_row["ap_form"],
        ls_xy=param_row["ls_xy"],
        ls_z=param_row["ls_z"],
        os_xyz=param_row["os_xyz"],
        ls_t=param_row["ls_t"],
        os_t=param_row["os_t"],
        ls_ap=param_row["ls_ap"],
        os_ap=param_row["os_ap"],
    )

    gp = GaussianProcess(m, K)
    gp.fit(X_train, y_train, obs_count_train, obs_sse=obs_sse_train)

    start = time.time()
    chain = gp.gibbs(
        key=jr.PRNGKey(KEY_SEED + 1000 * param_id + fold_idx),
        chains=N_CHAINS,
        samples=N_SAMPLES,
        calibration_iters=CALIBRATION_ITERS,
        jitter_factor=JITTER_FACTOR,
    )
    fit_elapsed = time.time() - start

    cov_chains = chain[1][:, BURNIN::THIN, :]
    start = time.time()
    pred_means, pred_vars = gp.predict(X_test, cov_chains, method=PREDICT_METHOD)
    predict_elapsed = time.time() - start

    y_hat = pred_means.mean(axis=0)
    mse = float(jnp.mean((y_test - y_hat) ** 2))

    row = {
        "param_id": param_id,
        "fold_idx": int(fold_idx),
        "n_train": int(X_train.shape[0]),
        "n_test": int(X_test.shape[0]),
        "mse": mse,
        "fit_elapsed": float(fit_elapsed),
        "predict_elapsed": float(predict_elapsed),
        "total_elapsed": float(time.time() - total_start),
        "jitter": float(gp.jitter),
        **{col: param_row[col] for col in ["ap_form", "ls_xy", "ls_z", "os_xyz", "ls_t", "os_t", "ls_ap", "os_ap"]},
    }
    write_result_row(out_path, row)
    return row


In [ ]:

# Expensive: stage-1 3-fold GP CV over all configs. Restartable.
stage1_split_dir = SPLIT_ROOT / "inner_geo3"
run_stage_loop("Stage 1", grid, STAGE1_FOLDS, stage1_split_dir, STAGE1_RESULT_DIR)


In [ ]:

stage1_results = completed_gp_results(STAGE1_RESULT_DIR)
stage1_summary = summarize_cv_results(stage1_results)
stage1_summary = stage1_summary.merge(grid, on="param_id", how="left")
stage1_complete = stage1_summary[stage1_summary["n_folds"] == STAGE1_FOLDS].copy()
stage1_summary.head(25)


In [ ]:

if len(stage1_complete) < STAGE2_TOP_N:
    raise RuntimeError(
        f"Need {STAGE2_TOP_N} complete stage-1 configs before stage 2; "
        f"found {len(stage1_complete)}."
    )
stage2_param_ids = stage1_complete.head(STAGE2_TOP_N)["param_id"].astype(int).to_list()
stage2_grid = grid[grid["param_id"].isin(stage2_param_ids)].copy()
stage2_grid


In [ ]:

# Expensive: stage-2 5-fold GP CV over the top 25 stage-1 configs. Restartable.
stage2_split_dir = SPLIT_ROOT / "inner_geo5"
run_stage_loop("Stage 2", stage2_grid, STAGE2_FOLDS, stage2_split_dir, STAGE2_RESULT_DIR)


In [ ]:

stage2_results = completed_gp_results(STAGE2_RESULT_DIR)
stage2_summary = summarize_cv_results(stage2_results)
stage2_summary = stage2_summary.merge(grid, on="param_id", how="left")
stage2_complete = stage2_summary[stage2_summary["n_folds"] == STAGE2_FOLDS].copy()
stage2_summary.head(25)


In [ ]:

if stage2_complete.empty:
    raise RuntimeError("Need at least one complete stage-2 config before final holdout evaluation.")
best_params = stage2_complete.iloc[0]
best_params


In [ ]:

def load_outer(prefix):
    return {
        "X": jnp.asarray(np.load(SPLIT_ROOT / f"{prefix}_X.npy")),
        "y": jnp.asarray(np.load(SPLIT_ROOT / f"{prefix}_y.npy")),
        "obs_count": jnp.asarray(np.load(SPLIT_ROOT / f"{prefix}_obs_count.npy")),
        "obs_sse": jnp.asarray(np.load(SPLIT_ROOT / f"{prefix}_obs_sse.npy")),
    }

outer_train = load_outer("outer_train")
outer_holdout = load_outer("outer_holdout")

m = make_indoor_outdoor_mean(outer_train["X"], outer_train["y"])
K = make_wifi_kernel(
    ap_form=best_params["ap_form"],
    ls_xy=best_params["ls_xy"],
    ls_z=best_params["ls_z"],
    os_xyz=best_params["os_xyz"],
    ls_t=best_params["ls_t"],
    os_t=best_params["os_t"],
    ls_ap=best_params["ls_ap"],
    os_ap=best_params["os_ap"],
)

gp = GaussianProcess(m, K)
gp.fit(outer_train["X"], outer_train["y"], outer_train["obs_count"], obs_sse=outer_train["obs_sse"])

start = time.time()
chain = gp.gibbs(
    key=jr.PRNGKey(KEY_SEED + 999_999),
    chains=N_CHAINS,
    samples=N_SAMPLES,
    calibration_iters=CALIBRATION_ITERS,
    jitter_factor=JITTER_FACTOR,
)
fit_elapsed = time.time() - start

cov_chains = chain[1][:, BURNIN::THIN, :]
start = time.time()
pred_means, pred_vars = gp.predict(outer_holdout["X"], cov_chains, method=PREDICT_METHOD)
predict_elapsed = time.time() - start

y_hat = pred_means.mean(axis=0)
final_mse = float(jnp.mean((outer_holdout["y"] - y_hat) ** 2))
final_row = {
    "model": "gp",
    "param_id": int(best_params["param_id"]),
    "mse": final_mse,
    "n_train": int(outer_train["X"].shape[0]),
    "n_test": int(outer_holdout["X"].shape[0]),
    "fit_elapsed": float(fit_elapsed),
    "predict_elapsed": float(predict_elapsed),
    "jitter": float(gp.jitter),
    **{col: best_params[col] for col in ["ap_form", "ls_xy", "ls_z", "os_xyz", "ls_t", "os_t", "ls_ap", "os_ap"]},
}
write_result_row(FINAL_RESULT_PATH, final_row)
final_row
